Stage 2 - PPO fine-tuning of the pretrained Decision Transformer.
 
* Actor-critic: the transformer's last token feeds mu (actor), value (critic) and the
  auxiliary future-return head (kept alive as a "world-model" regulariser).
* Gaussian policy over the env's [-1,1] action logits (state-independent log-std).
* Discount PPO_GAMMA (config.py) is where the "diminishing reward" lives.
* One rollout == one full training episode (EPISODE_LEN steps) per env; truncation is
  bootstrapped with the critic.
* Overfitting guards (your note: freeze/regularise the backbone):
    - bc_coef : MSE penalty keeping the policy mean close to the FROZEN pretrained policy
    - value warm-up: first --vf_warmup iterations only train the value head (backbone
      detached), so early advantages are not garbage
    - low lr, KL early stop, best checkpoint chosen on the VAL split


In [1]:
import copy
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import config as C
from trading_env import TradingEnv
from model.model import load_policy,save_policy
from common import *



# Constants

In [2]:
ckpt_dir = "checkpoints/dt_best.pt"
out_dir = "checkpoints/ppo"
iters = 150
n_envs = 16
n_steps = C.EPISODE_LEN
epochs = 4
minibatch = 512
lr = 3e-5
gamma =  C.PPO_GAMMA
λ = 0.95
clip = 0.2
entropy = 5e-4
vf = 0.5
bc_coef = 0.5
aux_coef = 0.5
target_kl = 0.03
rew_scale = 0.1
vf_warmup = 5
eval_every = 5
rho = None
news_train = None
news_val = None
seed = 0 
bf16 = 'store_true'



In [4]:
torch.manual_seed(seed)
np.random.default_rng(seed)
device = get_device()
os.makedirs(os.path.dirname(out_dir),exist_ok=True)


In [10]:
policy,meta = load_policy(ckpt_dir,device=device)
ref = copy.deepcopy(policy).eval()

for p in ref.parameters():
    p.requires_grad_(False)

rho_raw = rho if rho is not None else meta.get('rho_target')
rho_z = (rho_raw - meta['rho_mean']) / (meta['rho_std']+1e-8)
print(f"conditioning on rho={rho_raw:.4f} (z={rho_z:.2f})")

panel,news = load_panel('train'),load_news(news_train)
envs = [TradingEnv(panel,news,episode_len=C.EPISODE_LEN) for _ in range(n_envs)]
val_env = TradingEnv(load_panel('val'),news=load_news(news_val))

S,N = n_steps,n_envs
A = envs[0].action_space.shape[0]
n_assets,H = envs[0].n,len(envs[0].horizons)
rho_full =torch.full((N,),rho_z,device=device)



conditioning on rho=0.2556 (z=1.23)


In [ ]:
obs_list = [e.reset(seed=seed+i)[0] for i,e in enumerate(envs)]
o0 = to_torch(stack_obs(obs_list),device=device)
buf = {k: torch.zeros((S,) + tuple(v.shape), device=device) for k, v in o0.items()}
act_buf = torch.zeros(S, N, A, device=device)
logp_buf = torch.zeros(S, N, device=device)
val_buf = torch.zeros(S, N, device=device)
rew_buf = torch.zeros(S, N, device=device)
done_buf = torch.zeros(S, N, device=device)
aux_buf = torch.zeros(S, N, n_assets, H, device=device)
auxm_buf = torch.zeros(S, N, H, device=device)


In [25]:
from tqdm.auto import tqdm
import pprint
opt = torch.optim.RMSprop(policy.parameters(),lr=lr,weight_decay=0.0)
best = -1e9

for it in tqdm(range(1,iters+1),desc="Training"):
    policy.eval()
    ep_ret = np.zeros(N)
    for s in range(S):
        o = to_torch(stack_obs(obs_list),device=device)
        for k in buf:
            buf[k][s] = o[k]
        aux_pairs = [e._aux() for e in envs]
        aux_buf[s] = torch.as_tensor(np.stack([a for a, _ in aux_pairs]), device=device)
        auxm_buf[s] = torch.as_tensor(np.stack([m for _, m in aux_pairs]), device=device)
        with torch.no_grad(), amp_ctx(device,bf16):
            action,logp,_,value,_,_ = policy.get_action_and_value(o,rho_full)
        act_buf[s],logp_buf[s],val_buf[s] = action,logp,value
        act_np = action.cpu().numpy()
        rewards = np.zeros(N,np.float32)
        dones = np.zeros(N,np.float32)
        
        boot_idx,boot_obs,new_obs = [],[],[]
        for i,env in tqdm(enumerate(envs),desc=f"Step {s+1}/{S}",total=N,leave=False):
            obs,rew,term,trunc,_ = env.step(act_np[i])
            ep_ret[i] += rew
            rewards[i] = rew* rew_scale
            if term or trunc:
                dones[i] = 1.0
                if trunc and not term:
                    boot_idx.append(i)
                    boot_obs.append(obs)
                ob,_ = env.reset()
            new_obs.append(obs)
        if boot_idx:
            ob_t = to_torch(stack_obs(boot_obs),device=device)
            with torch.no_grad(), amp_ctx(device,bf16):
                 v_boot = policy.get_action_and_value(ob_t, rho_full[:len(boot_idx)])[3]
            rewards[boot_idx] += gamma * v_boot.cpu().numpy()
        rew_buf[s] = torch.as_tensor(rewards,device=device)
        done_buf[s] = torch.as_tensor(dones,device=device)
        
        obs_list = new_obs
        
    with torch.no_grad(), amp_ctx(device,bf16):
        v_last = policy.get_action_and_value(to_torch(stack_obs(obs_list),device=device), rho_full)[3]
    
    v_last = v_last.detach()
    
    adv = torch.zeros_like(rew_buf)
    last = torch.zeros(N,device=device)
    
    for t in reversed(range(S)):
        nxt = v_last if t == S-1 else val_buf[t+1]
        nonterminal = 1.0 - done_buf[t]
        delta = rew_buf[t] + gamma * nxt * nonterminal - val_buf[t]
        last = delta + gamma * λ * nonterminal * last
        adv[t] = last
    ret =  adv + val_buf
    
    B = S*N
    b_obs = {k: v.reshape((B,) + tuple(v.shape[2:])) for k, v in buf.items()}
    b_act,b_logp = act_buf.reshape(B,A),logp_buf.reshape(B)
    b_ret = ret.reshape(B)
    b_adv = adv.reshape(B)
    b_adv = (b_adv - b_adv.mean()) / (b_adv.std() + 1e-8)
    b_aux,b_auxm = aux_buf.reshape(B,n_assets,H),auxm_buf.reshape(B,H)
    b_rho = torch.full((B,), rho_z, device=device)
    
    
    warm = it <= vf_warmup
    policy.train()
    stats=  {"pg":[],"v":[],"kl":[],"bc":[],"ent":[]}
    stop = False
    for epoch in range(epochs):
        perm = torch.randperm(B,device=device)
        epoch_kl = []
        for start in range(0,B,minibatch):
            idx = perm[start:start+minibatch]
            mb_obs = {k: v[idx] for k, v in b_obs.items()}
            with amp_ctx(device, bf16):
                _, new_logp, ent, new_v, mu, aux = policy.get_action_and_value(mb_obs, b_rho[idx], action=b_act[idx], detach_value=warm)
                with torch.no_grad():
                    mu_ref = ref(mb_obs, b_rho[idx])[0]
            vloss = F.smooth_l1_loss(new_v, b_ret[idx],beta=1.0)
            if warm:
                loss = vf * vloss
                pg = bc = ent_m = torch.zeros(())
                kl = torch.zeros(())
            else:
                logr = new_logp - b_logp[idx]
                ratio = logr.exp()
                a_ = b_adv[idx]
                pg = torch.max(-a_*ratio, -a_ * ratio.clamp(1.0-clip,1.0+clip)).mean()
                bc = F.mse_loss(mu, mu_ref)
                ent_m = ent.mean()
                al = aux_loss_fn(aux,b_aux[idx],b_auxm[idx],policy.aux_scale)
                loss = (pg - ent_m + vf * vloss + bc_coef * bc + aux_coef * al)
                kl = ((ratio-1)-logr).mean().detach()
            # if not torch.isfinite(loss):
            #     print(f"Non-finite loss detected at epoch {epoch}, skipping update")
            #     continue
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(policy.parameters(),0.5)
            opt.step()
            for k_,v_ in zip(['pg','v','kl','bc','ent'],[pg,vloss,kl,bc,ent_m]):
                stats[k_].append(float(v_))
            epoch_kl.append(float(kl))
        if not warm and np.mean(epoch_kl) > 1.5 * target_kl:
            print(f"Early stopping at epoch {epoch} due to reaching max kl")
            stop = True
            break
    mean = {k :float(np.mean(v)) for k,v in stats.items()}
    print(f"it {it:4d} | ep_return {ep_ret.mean():7.2f} | pg {mean['pg']:+.4f} | " f"v {mean['v']:.4f} | kl {mean['kl']:.4f} | bc {mean['bc']:.4f} "f"ent {mean['ent']:.2f}{' | value warm-up' if warm else ''}" f"{' | early-stop(kl)' if stop else ''}")
    if it % eval_every == 0 or it == iters:
            print(f" Evaluating policy at iteration {it}")
            ev = evaluate_policy(policy, val_env, rho_z, device, label=f"VAL it {it}")
            score = ev["metrics"]["Agent"]["total_return"]
            if score > best:
                best = score
                save_policy(f"{out_dir}_best.pt", policy, meta)
                print(f"  ** new best val return {best:.1%} -> {out_dir}_best.pt")
    save_policy(f"{out_dir}_last.pt", policy, meta)
    
print("Done!")


Training:   1%|          | 1/150 [00:05<13:34,  5.46s/it]

it    1 | ep_return   24.82 | pg +0.0000 | v 0.1148 | kl 0.0000 | bc 0.0000 ent 0.00 | value warm-up


Training:   1%|▏         | 2/150 [00:10<13:00,  5.28s/it]

it    2 | ep_return   35.78 | pg +0.0000 | v 0.1248 | kl 0.0000 | bc 0.0000 ent 0.00 | value warm-up


Training:   2%|▏         | 3/150 [00:15<12:50,  5.24s/it]

...

Training:  99%|█████████▉| 149/150 [17:00<00:06,  6.43s/it]

Early stopping at epoch 2 due to reaching max kl
it  149 | ep_return   96.00 | pg +0.0004 | v 0.0296 | kl 0.0380 | bc 0.0689 ent 3.90 | early-stop(kl)


it  150 | ep_return   92.50 | pg -0.0080 | v 0.0270 | kl 0.0318 | bc 0.0636 ent 3.90
 Evaluating policy at iteration 150


Training: 100%|██████████| 150/150 [17:09<00:00,  6.86s/it]


== VAL it 150 2023-03-30 -> 2024-12-30 (440 steps) ==
                       return  sharpe   maxDD
  Agent                151.7%    1.66   24.7%
  SPY buy&hold          49.2%    1.94   10.0%
  Equal-weight B&H     127.9%    2.12   18.7%
  agent: avg cash 1% | total turnover 30.2 | total costs 0.0274
Done!
